# NBA Polymarket Backtest (Parquet Version) — Colab

In [ ]:
!pip install -q "numpy<2.0.0" "pandas==2.2.2" "scikit-learn==1.5.2" "xgboost==2.1.3" "nba-on-court>=0.2.0,<1.0" "joblib>=1.3" "matplotlib>=3.8" "tqdm>=4.66" "pyarrow"

import os
from google.colab import drive

# Mount Google Drive to save data persistently
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/nba_backtest_data'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Data directory ready at: {DRIVE_DIR}")

# Clone repository and install local module
REPO_PATH = '/content/nba_bot'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/cyruslayo/nba_bot.git {REPO_PATH}
else:
    !git -C {REPO_PATH} pull

!pip uninstall -y nba-bot > /dev/null 2>&1 || true
!pip install -e /content/nba_bot --no-deps -q

print('Setup complete')

In [ ]:
import requests
from datetime import datetime, timedelta

def download_pmxt_parquet(start_date, end_date, save_dir):
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    while current <= end:
        date_str = current.strftime("%Y-%m-%d")
        
        # Loop through 24 hours (00 to 23)
        for hour in range(24):
            hour_str = f"{hour:02d}"
            filename = f"polymarket_orderbook_{date_str}T{hour_str}.parquet"
            url = f"https://archive.pmxt.dev/Polymarket/{filename}"
            save_path = os.path.join(save_dir, filename)
            
            if not os.path.exists(save_path):
                print(f"Downloading {filename}...")
                response = requests.get(url, stream=True)
                
                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"Saved to {save_path}")
                else:
                    pass # Skip missing hours without erroring
            else:
                pass # Already exists
                
        current += timedelta(days=1)
    print("Download process finished.")

# Example: Download data for a specific day/week and save it to your Drive (Uncomment to execute)
# download_pmxt_parquet("2024-01-01", "2024-01-02", DRIVE_DIR)

In [ ]:
import json
import pandas as pd
import requests


def _safe_gamma_list(value):
    if isinstance(value, list):
        return value
    if value in (None, '', [], {}):
        return []
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []


def _as_utc_timestamp(value):
    ts = pd.to_datetime(value, utc=True, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    return ts


def _gamma_market_rows(events: list[dict]) -> list[dict]:
    rows = []
    for event in events:
        event_id = event.get('id')
        event_title = event.get('title')
        event_slug = event.get('slug')
        event_start_ts = _as_utc_timestamp(event.get('startDate'))
        for market in event.get('markets', []):
            outcomes = _safe_gamma_list(market.get('outcomes'))
            outcome_prices = _safe_gamma_list(market.get('outcomePrices'))
            clob_token_ids = _safe_gamma_list(market.get('clobTokenIds'))
            try:
                liquidity = float(market.get('liquidity', 0) or 0)
            except (TypeError, ValueError):
                liquidity = 0.0
            rows.append({
                'event_id': str(event_id) if event_id is not None else None,
                'event_title': event_title,
                'event_slug': event_slug,
                'event_start_ts': event_start_ts,
                'market_id': str(market.get('id')) if market.get('id') is not None else None,
                'question': market.get('question'),
                'outcomes': outcomes,
                'outcome_prices': outcome_prices,
                'clob_token_ids': clob_token_ids,
                'liquidity': liquidity,
                'active': market.get('active'),
                'closed': market.get('closed'),
            })
    return rows


# Find historical NBA markets
def search_nba_markets(start_date: str, end_date: str, limit: int = 100, closed: bool = True, active: bool = True, buffer_days: int = 1) -> pd.DataFrame:
    start_bound = pd.Timestamp(start_date)
    end_bound = pd.Timestamp(end_date)
    if start_bound.tzinfo is None:
        start_bound = start_bound.tz_localize('UTC')
    else:
        start_bound = start_bound.tz_convert('UTC')
    if end_bound.tzinfo is None:
        end_bound = end_bound.tz_localize('UTC')
    else:
        end_bound = end_bound.tz_convert('UTC')
    start_bound = start_bound.normalize() - pd.Timedelta(days=buffer_days)
    end_bound = end_bound.normalize() + pd.Timedelta(days=buffer_days + 1) - pd.Timedelta(microseconds=1)

    status_filters = []
    if closed:
        status_filters.append({'active': 'false', 'closed': 'true'})
    if active:
        status_filters.append({'active': 'true', 'closed': 'false'})
    if not status_filters:
        status_filters.append({})

    seen_market_ids: set[str] = set()
    all_events: list[dict] = []

    for status_filter in status_filters:
        offset = 0
        while True:
            params = {
                'series_id': 10345,
                'tag_id': 100639,
                'limit': limit,
                'offset': offset,
            }
            params.update(status_filter)
            response = requests.get('https://gamma-api.polymarket.com/events', params=params, timeout=20)
            response.raise_for_status()
            page_events = response.json()
            if not page_events:
                break
            all_events.extend(page_events)
            if len(page_events) < limit:
                break
            offset += limit

    markets_df = pd.DataFrame(_gamma_market_rows(all_events))
    if markets_df.empty:
        return markets_df

    markets_df = markets_df[markets_df['market_id'].notna()].copy()
    markets_df['event_start_ts'] = pd.to_datetime(markets_df['event_start_ts'], utc=True, errors='coerce')
    markets_df = markets_df[markets_df['event_start_ts'].notna()].copy()
    markets_df = markets_df[(markets_df['event_start_ts'] >= start_bound) & (markets_df['event_start_ts'] <= end_bound)].copy()

    deduped_rows = []
    for row in markets_df.sort_values(['event_start_ts', 'market_id']).to_dict('records'):
        market_id = str(row['market_id'])
        if market_id in seen_market_ids:
            continue
        seen_market_ids.add(market_id)
        deduped_rows.append(row)

    return pd.DataFrame(deduped_rows).reset_index(drop=True)

In [ ]:
import ast
import json
import math
import os
import re
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import pyarrow.parquet as pq

from nba_bot.features import STARTTYPE_FALLBACK, _encode_starttype, build_game_state_rows
from nba_bot.model import load_model, predict_home_win_prob

MODEL_PATH = '/content/drive/MyDrive/nba_bot/xgb_model_t2.pkl'
PMXT_ROOT = Path(DRIVE_DIR)
PMXT_GLOB = '*.parquet'
NBA_SEASONS = [2024]
USE_ADVANCED_FEATURES = True
INITIAL_BANKROLL = 1000.0
LATENCY_BUFFER_MS = 500
COOLDOWN_MS = 1500
MIN_TARGET_DELTA_USDC = 5.0
MIN_DIVERGENCE_RATIO = 0.10
MIN_RESTING_DEPTH_USDC = 50.0
ADVERSE_SELECTION_PENALTY = 0.01
FEE_RATE = 0.02
PER_GAME_BANKROLL_DIVISOR = 5.0
HOLDOUT_PCT = 0.30
CHUNK_SIZE = 100000
MARKET_DATE_BUFFER_DAYS = 1
MATCH_SCORE_MIN = 8.0
MATCH_SCORE_GAP_MIN = 2.5
MAX_TIPOFF_DELTA_HOURS = 18.0
LOOKUP_CACHE_DIR = Path(DRIVE_DIR) / 'market_lookup_cache'
LOOKUP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REUSE_SAVED_LOOKUP = True

NBA_TEAM_DATA = {
    1610612737: {'city': 'Atlanta', 'team': 'Hawks', 'abbr': 'ATL', 'aliases': ['atlanta hawks', 'hawks', 'atlanta', 'atl']},
    1610612738: {'city': 'Boston', 'team': 'Celtics', 'abbr': 'BOS', 'aliases': ['boston celtics', 'celtics', 'boston', 'bos']},
    1610612751: {'city': 'Brooklyn', 'team': 'Nets', 'abbr': 'BKN', 'aliases': ['brooklyn nets', 'nets', 'brooklyn', 'bkn']},
    1610612766: {'city': 'Charlotte', 'team': 'Hornets', 'abbr': 'CHA', 'aliases': ['charlotte hornets', 'hornets', 'charlotte', 'cha']},
    1610612741: {'city': 'Chicago', 'team': 'Bulls', 'abbr': 'CHI', 'aliases': ['chicago bulls', 'bulls', 'chicago', 'chi']},
    1610612739: {'city': 'Cleveland', 'team': 'Cavaliers', 'abbr': 'CLE', 'aliases': ['cleveland cavaliers', 'cavaliers', 'cavs', 'cleveland', 'cle']},
    1610612742: {'city': 'Dallas', 'team': 'Mavericks', 'abbr': 'DAL', 'aliases': ['dallas mavericks', 'mavericks', 'mavs', 'dallas', 'dal']},
    1610612743: {'city': 'Denver', 'team': 'Nuggets', 'abbr': 'DEN', 'aliases': ['denver nuggets', 'nuggets', 'denver', 'den']},
    1610612765: {'city': 'Detroit', 'team': 'Pistons', 'abbr': 'DET', 'aliases': ['detroit pistons', 'pistons', 'detroit', 'det']},
    1610612744: {'city': 'Golden State', 'team': 'Warriors', 'abbr': 'GSW', 'aliases': ['golden state warriors', 'warriors', 'golden state', 'gsw']},
    1610612745: {'city': 'Houston', 'team': 'Rockets', 'abbr': 'HOU', 'aliases': ['houston rockets', 'rockets', 'houston', 'hou']},
    1610612754: {'city': 'Indiana', 'team': 'Pacers', 'abbr': 'IND', 'aliases': ['indiana pacers', 'pacers', 'indiana', 'ind']},
    1610612746: {'city': 'Los Angeles', 'team': 'Clippers', 'abbr': 'LAC', 'aliases': ['los angeles clippers', 'la clippers', 'clippers', 'lac']},
    1610612747: {'city': 'Los Angeles', 'team': 'Lakers', 'abbr': 'LAL', 'aliases': ['los angeles lakers', 'la lakers', 'lakers', 'lal']},
    1610612763: {'city': 'Memphis', 'team': 'Grizzlies', 'abbr': 'MEM', 'aliases': ['memphis grizzlies', 'grizzlies', 'memphis', 'mem']},
    1610612748: {'city': 'Miami', 'team': 'Heat', 'abbr': 'MIA', 'aliases': ['miami heat', 'heat', 'miami', 'mia']},
    1610612749: {'city': 'Milwaukee', 'team': 'Bucks', 'abbr': 'MIL', 'aliases': ['milwaukee bucks', 'bucks', 'milwaukee', 'mil']},
    1610612750: {'city': 'Minnesota', 'team': 'Timberwolves', 'abbr': 'MIN', 'aliases': ['minnesota timberwolves', 'timberwolves', 'wolves', 'minnesota', 'min']},
    1610612740: {'city': 'New Orleans', 'team': 'Pelicans', 'abbr': 'NOP', 'aliases': ['new orleans pelicans', 'pelicans', 'new orleans', 'nop', 'nola']},
    1610612752: {'city': 'New York', 'team': 'Knicks', 'abbr': 'NYK', 'aliases': ['new york knicks', 'knicks', 'new york', 'nyk']},
    1610612760: {'city': 'Oklahoma City', 'team': 'Thunder', 'abbr': 'OKC', 'aliases': ['oklahoma city thunder', 'thunder', 'oklahoma city', 'okc']},
    1610612753: {'city': 'Orlando', 'team': 'Magic', 'abbr': 'ORL', 'aliases': ['orlando magic', 'magic', 'orlando', 'orl']},
    1610612755: {'city': 'Philadelphia', 'team': '76ers', 'abbr': 'PHI', 'aliases': ['philadelphia 76ers', '76ers', 'sixers', 'philadelphia', 'phi']},
    1610612756: {'city': 'Phoenix', 'team': 'Suns', 'abbr': 'PHX', 'aliases': ['phoenix suns', 'suns', 'phoenix', 'phx']},
    1610612757: {'city': 'Portland', 'team': 'Trail Blazers', 'abbr': 'POR', 'aliases': ['portland trail blazers', 'trail blazers', 'blazers', 'portland', 'por']},
    1610612758: {'city': 'Sacramento', 'team': 'Kings', 'abbr': 'SAC', 'aliases': ['sacramento kings', 'kings', 'sacramento', 'sac']},
    1610612759: {'city': 'San Antonio', 'team': 'Spurs', 'abbr': 'SAS', 'aliases': ['san antonio spurs', 'spurs', 'san antonio', 'sas']},
    1610612761: {'city': 'Toronto', 'team': 'Raptors', 'abbr': 'TOR', 'aliases': ['toronto raptors', 'raptors', 'toronto', 'tor']},
    1610612762: {'city': 'Utah', 'team': 'Jazz', 'abbr': 'UTA', 'aliases': ['utah jazz', 'jazz', 'utah', 'uta']},
    1610612764: {'city': 'Washington', 'team': 'Wizards', 'abbr': 'WAS', 'aliases': ['washington wizards', 'wizards', 'washington', 'was']},
}

MARKET_TO_GAME: dict[str, Any] = {}
MARKET_LOOKUP = pd.DataFrame()
MARKET_MATCHES = pd.DataFrame()
MATCH_DIAGNOSTICS: dict[str, pd.DataFrame] = {}

BACKTEST_STATE_PATH = Path('/content/backtest_state.json')

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
def fix_columns(df: pd.DataFrame | None) -> pd.DataFrame | None:
    if df is None: return None
    out = df.copy()
    out.columns = [str(c).upper() for c in out.columns]
    if 'GAME_ID' not in out.columns:
        candidates = [c for c in out.columns if 'GAMEID' in c or 'GAME_ID' in c or 'NBASTATS' in c]
        if candidates: out = out.rename(columns={candidates[0]: 'GAME_ID'})
    return out


def load_data_safe(seasons: list[int], data_type: str) -> pd.DataFrame | None:
    import nba_on_court as noc
    try:
        loaded = noc.load_nba_data(seasons=seasons, data=data_type)
        loaded = fix_columns(loaded)
        if loaded is not None and not loaded.empty: return loaded
    except Exception as exc:
        print(f'Library load failed for {data_type}: {exc}')
    frames = []
    for season in seasons:
        path = Path(f'/content/{data_type}_{season}.tar.xz')
        if path.exists():
            frame = pd.read_csv(path, compression='xz')
            frame = fix_columns(frame)
            frames.append(frame)
    if not frames: return None
    return pd.concat(frames, ignore_index=True)


def load_player_ratings(enabled: bool) -> dict[int, float]:
    if not enabled: return {}
    try:
        from nba_bot.team_stats_cache import refresh_team_stats
        return refresh_team_stats(output_path='/content/team_stats.json')
    except Exception as exc:
        print(f'Player ratings unavailable, falling back to defaults: {exc}')
        return {}


def _candidate_column(columns: list[str], names: list[str]) -> str | None:
    lowered = {str(c).lower(): c for c in columns}
    for name in names:
        if name.lower() in lowered: return lowered[name.lower()]
    for column in columns:
        if any(name.lower() in str(column).lower() for name in names): return column
    return None


def _parse_score_value(value: Any) -> tuple[float, float]:
    text = '' if pd.isna(value) else str(value)
    if ' - ' in text:
        left, right = text.split(' - ', 1)
        try: return float(right), float(left)
        except ValueError: return math.nan, math.nan
    return math.nan, math.nan


def _combine_event_time(frame: pd.DataFrame) -> pd.Series:
    timestamp_col = _candidate_column(list(frame.columns), ['event_time', 'timestamp', 'ts', 'created_at', 'time'])
    if timestamp_col is not None:
        parsed = pd.to_datetime(frame[timestamp_col], utc=True, errors='coerce')
        if parsed.notna().any(): return parsed
    date_col = _candidate_column(list(frame.columns), ['GAME_DATE_EST', 'GAME_DATE', 'DATE'])
    wc_col = _candidate_column(list(frame.columns), ['WCTIMESTRING', 'wall_clock'])
    if date_col is not None and wc_col is not None:
        parsed = pd.to_datetime(frame[date_col].astype(str) + ' ' + frame[wc_col].astype(str), utc=True, errors='coerce')
        if parsed.notna().any(): return parsed
    return pd.to_datetime(pd.Series(np.arange(len(frame))), unit='s', utc=True, errors='coerce')


def _infer_team_ids(game_df: pd.DataFrame) -> tuple[Any, Any]:
    home_team_id, away_team_id = None, None
    for col in ['HOME_TEAM_ID', 'HTEAM_ID', 'HOME_ID']:
        if col in game_df.columns and game_df[col].notna().any():
            home_team_id = game_df[col].dropna().iloc[0]
            break
    for col in ['VISITOR_TEAM_ID', 'VTEAM_ID', 'AWAY_ID']:
        if col in game_df.columns and game_df[col].notna().any():
            away_team_id = game_df[col].dropna().iloc[0]
            break
    if home_team_id is None and 'HOMEDESCRIPTION' in game_df.columns and 'PLAYER1_TEAM_ID' in game_df.columns:
        mask = game_df['HOMEDESCRIPTION'].notna() & game_df['PLAYER1_TEAM_ID'].notna()
        if mask.any(): home_team_id = game_df.loc[mask, 'PLAYER1_TEAM_ID'].iloc[0]
    if away_team_id is None and 'VISITORDESCRIPTION' in game_df.columns and 'PLAYER1_TEAM_ID' in game_df.columns:
        mask = game_df['VISITORDESCRIPTION'].notna() & game_df['PLAYER1_TEAM_ID'].notna()
        if mask.any(): away_team_id = game_df.loc[mask, 'PLAYER1_TEAM_ID'].iloc[0]
    return home_team_id, away_team_id


def _normalize_market_text(value: Any) -> str:
    cleaned = re.sub(r'[^a-z0-9]+', ' ', str(value or '').lower())
    return re.sub(r'\s+', ' ', cleaned).strip()


def _team_meta(team_id: Any) -> dict[str, Any]:
    try:
        team_key = int(team_id)
    except (TypeError, ValueError):
        return {'city': '', 'team': '', 'abbr': '', 'aliases': []}
    meta = NBA_TEAM_DATA.get(team_key, {'city': '', 'team': '', 'abbr': '', 'aliases': []}).copy()
    aliases = set(meta.get('aliases', []))
    if meta.get('city') and meta.get('team'):
        aliases.add(f"{meta['city']} {meta['team']}")
    if meta.get('city'):
        aliases.add(meta['city'])
    if meta.get('team'):
        aliases.add(meta['team'])
    if meta.get('abbr'):
        aliases.add(meta['abbr'])
    meta['aliases'] = sorted({_normalize_market_text(alias) for alias in aliases if _normalize_market_text(alias)})
    return meta


def _team_aliases_for_row(game_row: pd.Series, side: str) -> set[str]:
    value = game_row.get(f'{side}_aliases', [])
    if isinstance(value, str):
        value = [value]
    aliases = {_normalize_market_text(item) for item in (value or []) if _normalize_market_text(item)}
    return {alias for alias in aliases if len(alias) >= 3}


def _match_count(text: str, aliases: set[str]) -> int:
    normalized = _normalize_market_text(text)
    return sum(1 for alias in aliases if alias and (alias in normalized or normalized in alias))


def _title_has_vs(title: Any) -> bool:
    normalized = _normalize_market_text(title)
    return ' vs ' in f' {normalized} ' or ' v ' in f' {normalized} '


def _infer_yes_team_side(question: Any, outcomes: Any, home_aliases: set[str], away_aliases: set[str]) -> str | None:
    question_text = _normalize_market_text(question)
    question_home = _match_count(question_text, home_aliases)
    question_away = _match_count(question_text, away_aliases)
    if question_home > question_away:
        return 'home'
    if question_away > question_home:
        return 'away'
    outcome_home_hits = []
    outcome_away_hits = []
    for idx, outcome in enumerate(outcomes or []):
        outcome_text = _normalize_market_text(outcome)
        outcome_home_hits.append((idx, _match_count(outcome_text, home_aliases)))
        outcome_away_hits.append((idx, _match_count(outcome_text, away_aliases)))
    home_candidates = [idx for idx, count in outcome_home_hits[:2] if count > 0]
    away_candidates = [idx for idx, count in outcome_away_hits[:2] if count > 0]
    if len(home_candidates) == 1 and len(away_candidates) == 1 and home_candidates[0] != away_candidates[0]:
        return 'home' if home_candidates[0] == 0 else 'away'
    return None


def build_game_catalog(df_nbastats: pd.DataFrame) -> pd.DataFrame:
    df = fix_columns(df_nbastats)
    if df is None or df.empty:
        raise ValueError('NBA stats data is empty; unable to build game catalog')
    date_col = _candidate_column(list(df.columns), ['GAME_DATE_EST', 'GAME_DATE', 'DATE'])
    rows = []
    for game_id, game_df in tqdm(df.groupby('GAME_ID'), desc='Building game catalog'):
        game_df = game_df.copy()
        event_times = _combine_event_time(game_df)
        tipoff_time = event_times.dropna().min() if event_times.notna().any() else pd.NaT
        home_team_id, away_team_id = _infer_team_ids(game_df)
        home_meta = _team_meta(home_team_id)
        away_meta = _team_meta(away_team_id)
        game_date = pd.NaT
        if date_col is not None and date_col in game_df.columns:
            parsed_date = pd.to_datetime(game_df[date_col].iloc[0], errors='coerce')
            if pd.notna(parsed_date):
                game_date = parsed_date.normalize()
        if pd.isna(game_date) and pd.notna(tipoff_time):
            tipoff_ts = pd.Timestamp(tipoff_time)
            if tipoff_ts.tzinfo is None:
                tipoff_ts = tipoff_ts.tz_localize('UTC')
            else:
                tipoff_ts = tipoff_ts.tz_convert('UTC')
            game_date = tipoff_ts.tz_convert('US/Eastern').tz_localize(None).normalize()
        rows.append({
            'game_id': str(game_id),
            'game_date': game_date,
            'tipoff_time': tipoff_time,
            'home_team_id': home_team_id,
            'away_team_id': away_team_id,
            'home_city': home_meta.get('city', ''),
            'home_team': home_meta.get('team', ''),
            'home_abbr': home_meta.get('abbr', ''),
            'home_aliases': home_meta.get('aliases', []),
            'away_city': away_meta.get('city', ''),
            'away_team': away_meta.get('team', ''),
            'away_abbr': away_meta.get('abbr', ''),
            'away_aliases': away_meta.get('aliases', []),
        })
    catalog = pd.DataFrame(rows)
    if catalog.empty:
        raise ValueError('No games were found in the NBA stats data')
    return catalog.sort_values(['game_date', 'game_id']).reset_index(drop=True)


def infer_pmxt_date_range(paths: list[Path]) -> tuple[str | None, str | None]:
    dates = []
    for path in paths:
        match = re.search(r'(\d{4}-\d{2}-\d{2})T\d{2}', path.name)
        if match:
            dates.append(pd.Timestamp(match.group(1)))
    if not dates:
        return None, None
    return min(dates).strftime('%Y-%m-%d'), max(dates).strftime('%Y-%m-%d')


def build_lookup_cache_key(pmxt_start_date: str, pmxt_end_date: str, seasons: list[int], file_count: int) -> str:
    season_text = '-'.join(str(season) for season in sorted(seasons))
    return f'{pmxt_start_date}_{pmxt_end_date}_seasons-{season_text}_files-{file_count}'


def lookup_cache_paths(cache_key: str) -> dict[str, Path]:
    base = LOOKUP_CACHE_DIR / cache_key
    return {
        'lookup': base.with_name(base.name + '_lookup.parquet'),
        'matches': base.with_name(base.name + '_matches.parquet'),
        'games': base.with_name(base.name + '_unmatched_games.parquet'),
        'markets': base.with_name(base.name + '_unmatched_markets.parquet'),
        'ambiguous': base.with_name(base.name + '_ambiguous_candidates.parquet'),
    }


def save_lookup_cache(cache_key: str, market_lookup: pd.DataFrame, market_matches: pd.DataFrame, diagnostics: dict[str, pd.DataFrame]) -> dict[str, Path]:
    paths = lookup_cache_paths(cache_key)
    market_lookup.to_parquet(paths['lookup'], index=False)
    market_matches.to_parquet(paths['matches'], index=False)
    diagnostics.get('unmatched_games', pd.DataFrame()).to_parquet(paths['games'], index=False)
    diagnostics.get('unmatched_markets', pd.DataFrame()).to_parquet(paths['markets'], index=False)
    diagnostics.get('ambiguous_candidates', pd.DataFrame()).to_parquet(paths['ambiguous'], index=False)
    return paths


def load_lookup_cache(cache_key: str) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, pd.DataFrame]] | tuple[None, None, None]:
    paths = lookup_cache_paths(cache_key)
    if not paths['lookup'].exists() or not paths['matches'].exists():
        return None, None, None
    market_lookup = pd.read_parquet(paths['lookup'])
    market_matches = pd.read_parquet(paths['matches'])
    diagnostics = {
        'unmatched_games': pd.read_parquet(paths['games']) if paths['games'].exists() else pd.DataFrame(),
        'unmatched_markets': pd.read_parquet(paths['markets']) if paths['markets'].exists() else pd.DataFrame(),
        'ambiguous_candidates': pd.read_parquet(paths['ambiguous']) if paths['ambiguous'].exists() else pd.DataFrame(),
    }
    return market_lookup, market_matches, diagnostics


def match_gamma_markets_to_games(markets_df: pd.DataFrame, game_catalog: pd.DataFrame, buffer_days: int = 1) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    empty_diagnostics = {
        'unmatched_games': pd.DataFrame(),
        'unmatched_markets': pd.DataFrame(),
        'ambiguous_candidates': pd.DataFrame(),
        'all_candidates': pd.DataFrame(),
    }
    if markets_df is None or markets_df.empty or game_catalog is None or game_catalog.empty:
        return pd.DataFrame(), empty_diagnostics

    candidate_rows = []
    for market in markets_df.to_dict('records'):
        event_start_ts = pd.to_datetime(market.get('event_start_ts'), utc=True, errors='coerce')
        event_date_local = pd.NaT
        if pd.notna(event_start_ts):
            event_date_local = event_start_ts.tz_convert('US/Eastern').tz_localize(None).normalize()
        candidates = game_catalog.copy()
        if pd.notna(event_date_local):
            lower = event_date_local - pd.Timedelta(days=buffer_days)
            upper = event_date_local + pd.Timedelta(days=buffer_days)
            candidates = candidates[(candidates['game_date'].notna()) & (candidates['game_date'] >= lower) & (candidates['game_date'] <= upper)].copy()
        if candidates.empty:
            continue

        title_text = _normalize_market_text(market.get('event_title'))
        question_text = _normalize_market_text(market.get('question'))
        outcome_text = ' '.join(_normalize_market_text(item) for item in (market.get('outcomes') or []))
        combined_text = ' '.join(part for part in [title_text, question_text, outcome_text] if part).strip()
        title_has_vs = _title_has_vs(market.get('event_title'))

        for _, game_row in candidates.iterrows():
            home_aliases = _team_aliases_for_row(game_row, 'home')
            away_aliases = _team_aliases_for_row(game_row, 'away')
            home_hits_title = _match_count(title_text, home_aliases)
            away_hits_title = _match_count(title_text, away_aliases)
            home_hits_question = _match_count(question_text, home_aliases)
            away_hits_question = _match_count(question_text, away_aliases)
            home_hits_outcomes = _match_count(outcome_text, home_aliases)
            away_hits_outcomes = _match_count(outcome_text, away_aliases)
            home_hits_total = _match_count(combined_text, home_aliases)
            away_hits_total = _match_count(combined_text, away_aliases)
            if home_hits_total <= 0 or away_hits_total <= 0:
                continue
            if home_hits_title <= 0 or away_hits_title <= 0:
                continue
            yes_team_side = _infer_yes_team_side(market.get('question'), market.get('outcomes'), home_aliases, away_aliases)
            time_delta_hours = math.inf
            if pd.notna(event_start_ts) and pd.notna(game_row.get('tipoff_time')):
                tipoff_ts = pd.to_datetime(game_row['tipoff_time'], utc=True, errors='coerce')
                if pd.notna(tipoff_ts):
                    time_delta_hours = abs((event_start_ts - tipoff_ts).total_seconds()) / 3600.0
            score = 0.0
            score += 3.0 * float(home_hits_title + away_hits_title)
            score += 1.25 * float(home_hits_question + away_hits_question)
            score += 0.75 * float(home_hits_outcomes + away_hits_outcomes)
            if title_has_vs:
                score += 2.0
            if yes_team_side is not None:
                score += 2.0
            if pd.notna(event_date_local) and pd.notna(game_row.get('game_date')) and event_date_local == game_row['game_date']:
                score += 3.0
            if math.isfinite(time_delta_hours):
                if time_delta_hours <= 4:
                    score += 5.0
                elif time_delta_hours <= 8:
                    score += 3.0
                elif time_delta_hours <= 12:
                    score += 1.0
                elif time_delta_hours > MAX_TIPOFF_DELTA_HOURS:
                    score -= 6.0
            strict_ok = bool(
                home_hits_title > 0 and
                away_hits_title > 0 and
                title_has_vs and
                yes_team_side is not None and
                math.isfinite(time_delta_hours) and
                time_delta_hours <= MAX_TIPOFF_DELTA_HOURS
            )
            candidate_rows.append({
                'market_id': str(market.get('market_id')),
                'event_id': market.get('event_id'),
                'event_slug': market.get('event_slug'),
                'event_title': market.get('event_title'),
                'event_start_ts': event_start_ts,
                'question': market.get('question'),
                'outcomes': market.get('outcomes'),
                'liquidity': float(market.get('liquidity') or 0.0),
                'game_id': str(game_row['game_id']),
                'game_date': game_row['game_date'],
                'tipoff_time': game_row['tipoff_time'],
                'home_team': game_row['home_team'],
                'away_team': game_row['away_team'],
                'home_city': game_row['home_city'],
                'away_city': game_row['away_city'],
                'yes_team_side': yes_team_side,
                'score': score,
                'time_delta_hours': time_delta_hours,
                'home_hits_title': home_hits_title,
                'away_hits_title': away_hits_title,
                'home_hits_question': home_hits_question,
                'away_hits_question': away_hits_question,
                'home_hits_outcomes': home_hits_outcomes,
                'away_hits_outcomes': away_hits_outcomes,
                'strict_ok': strict_ok,
                'title_has_vs': title_has_vs,
            })

    candidates_df = pd.DataFrame(candidate_rows)
    if candidates_df.empty:
        unmatched_games = game_catalog.copy()
        unmatched_markets = markets_df.copy()
        diagnostics = {
            'unmatched_games': unmatched_games.reset_index(drop=True),
            'unmatched_markets': unmatched_markets.reset_index(drop=True),
            'ambiguous_candidates': pd.DataFrame(),
            'all_candidates': candidates_df,
        }
        return pd.DataFrame(), diagnostics

    candidates_df = candidates_df[candidates_df['strict_ok']].copy()
    if candidates_df.empty:
        unmatched_games = game_catalog.copy()
        unmatched_markets = markets_df.copy()
        diagnostics = {
            'unmatched_games': unmatched_games.reset_index(drop=True),
            'unmatched_markets': unmatched_markets.reset_index(drop=True),
            'ambiguous_candidates': pd.DataFrame(),
            'all_candidates': candidates_df,
        }
        return pd.DataFrame(), diagnostics

    candidates_df = candidates_df.sort_values(['market_id', 'score', 'liquidity', 'time_delta_hours'], ascending=[True, False, False, True]).reset_index(drop=True)
    top_two = candidates_df.groupby('market_id').head(2).copy()
    rank_in_market = top_two.groupby('market_id').cumcount() + 1
    top_two['candidate_rank'] = rank_in_market
    top_one = top_two[top_two['candidate_rank'] == 1].copy()
    top_two_lookup = top_two[['market_id', 'candidate_rank', 'game_id', 'score']].copy()
    second_best = top_two_lookup[top_two_lookup['candidate_rank'] == 2].rename(columns={'game_id': 'second_game_id', 'score': 'second_score'})
    top_one = top_one.merge(second_best[['market_id', 'second_game_id', 'second_score']], on='market_id', how='left')
    top_one['score_gap'] = top_one['score'] - top_one['second_score'].fillna(-math.inf)
    top_one['score_gap'] = top_one['score_gap'].replace([math.inf], 9999.0)
    viable = top_one[(top_one['score'] >= MATCH_SCORE_MIN) & (top_one['score_gap'] >= MATCH_SCORE_GAP_MIN)].copy()

    viable = viable.sort_values(['score', 'liquidity', 'time_delta_hours'], ascending=[False, False, True]).reset_index(drop=True)
    chosen_rows = []
    used_games = set()
    used_markets = set()
    for row in viable.to_dict('records'):
        market_id = str(row['market_id'])
        game_id = str(row['game_id'])
        if market_id in used_markets or game_id in used_games:
            continue
        used_markets.add(market_id)
        used_games.add(game_id)
        chosen_rows.append(row)

    matches_df = pd.DataFrame(chosen_rows)
    if matches_df.empty:
        unmatched_games = game_catalog.copy()
        unmatched_markets = markets_df.copy()
    else:
        matched_game_ids = set(matches_df['game_id'].astype(str))
        matched_market_ids = set(matches_df['market_id'].astype(str))
        unmatched_games = game_catalog[~game_catalog['game_id'].astype(str).isin(matched_game_ids)].copy()
        unmatched_markets = markets_df[~markets_df['market_id'].astype(str).isin(matched_market_ids)].copy()

    ambiguous_market_ids = set(top_one[(top_one['score'] >= MATCH_SCORE_MIN) & (top_one['score_gap'] < MATCH_SCORE_GAP_MIN)]['market_id'].astype(str))
    ambiguous_candidates = candidates_df[candidates_df['market_id'].astype(str).isin(ambiguous_market_ids)].copy()

    diagnostics = {
        'unmatched_games': unmatched_games.sort_values(['game_date', 'game_id']).reset_index(drop=True),
        'unmatched_markets': unmatched_markets.sort_values(['event_start_ts', 'market_id']).reset_index(drop=True),
        'ambiguous_candidates': ambiguous_candidates.sort_values(['market_id', 'score'], ascending=[True, False]).reset_index(drop=True),
        'all_candidates': candidates_df.sort_values(['market_id', 'score'], ascending=[True, False]).reset_index(drop=True),
    }
    return matches_df.sort_values(['game_date', 'event_start_ts', 'game_id']).reset_index(drop=True), diagnostics


def build_market_lookup(matches: pd.DataFrame) -> pd.DataFrame:
    if matches is None or matches.empty:
        return pd.DataFrame(columns=['lookup_key', 'game_id', 'yes_team_side', 'market_id', 'event_slug'])
    lookup_rows = []
    for row in matches.to_dict('records'):
        keys = {
            str(row.get('market_id') or ''),
            str(row.get('event_slug') or ''),
        }
        for key in keys:
            normalized_key = str(key).strip()
            if not normalized_key or normalized_key.lower() == 'none' or normalized_key.lower() == 'nan':
                continue
            lookup_rows.append({
                'lookup_key': normalized_key,
                'game_id': str(row['game_id']),
                'yes_team_side': str(row.get('yes_team_side') or 'home'),
                'market_id': str(row.get('market_id') or normalized_key),
                'event_slug': row.get('event_slug'),
            })
    return pd.DataFrame(lookup_rows).drop_duplicates(subset=['lookup_key']).reset_index(drop=True)


def prepare_pbp_timeline(df_nbastats: pd.DataFrame, df_pbpstats: pd.DataFrame | None = None, player_ratings: dict[int, float] | None = None) -> pd.DataFrame:
    df = fix_columns(df_nbastats)
    pbp_lookup = fix_columns(df_pbpstats) if df_pbpstats is not None else None
    timeline_rows = []

    for game_id, game_df in tqdm(df.groupby('GAME_ID'), desc='Preparing PBP timeline'):
        game_df = game_df.sort_values(['PERIOD', 'PCTIMESTRING'], ascending=[True, False]).copy()
        game_df['event_time'] = _combine_event_time(game_df)
        home_team_id, away_team_id = _infer_team_ids(game_df)

        pq_home = float(player_ratings.get(int(home_team_id), 0.0)) if player_ratings and home_team_id is not None else 0.0
        pq_away = float(player_ratings.get(int(away_team_id), 0.0)) if player_ratings and away_team_id is not None else 0.0

        pbp_map = {}
        if pbp_lookup is not None and 'GAME_ID' in pbp_lookup.columns:
            game_pbp = pbp_lookup[pbp_lookup['GAME_ID'] == game_id].copy()
            if not game_pbp.empty:
                if 'PCTIMESTRING' not in game_pbp.columns and 'STARTTIME' in game_pbp.columns:
                    game_pbp = game_pbp.rename(columns={'STARTTIME': 'PCTIMESTRING'})
                if 'PCTIMESTRING' in game_pbp.columns:
                    game_pbp['_secs'] = game_pbp['PCTIMESTRING'].astype(str).str.split(':').apply(lambda parts: int(parts[0]) * 60 + int(parts[1]) if len(parts) == 2 else 0)
                    for row in game_pbp.itertuples():
                        period = int(getattr(row, 'PERIOD', 1) or 1)
                        pbp_map.setdefault(period, []).append((int(getattr(row, '_secs', 0) or 0), getattr(row, 'STARTTYPE', None)))

        home_score, away_score = 0.0, 0.0
        for play in game_df.itertuples(index=False):
            parsed_h, parsed_a = _parse_score_value(getattr(play, 'SCORE', None))
            if not math.isnan(parsed_h): home_score = parsed_h
            if not math.isnan(parsed_a): away_score = parsed_a

            period = int(getattr(play, 'PERIOD', 1) or 1)
            clock = str(getattr(play, 'PCTIMESTRING', '12:00'))
            clock_parts = clock.split(':')
            secs = int(clock_parts[0]) * 60 + int(clock_parts[1]) if len(clock_parts) == 2 else 720
            time_remaining = float((4 - period) * 720 + secs) if period <= 4 else float(max(0, secs))

            st_enc = STARTTYPE_FALLBACK
            for pbp_secs, starttype in pbp_map.get(period, []):
                if abs(pbp_secs - secs) <= 2:
                    st_enc = _encode_starttype(str(starttype))
                    break

            timeline_rows.append({
                'game_id': str(game_id),
                'event_time': getattr(play, 'event_time'),
                'period': period,
                'clock': clock,
                'home_score': float(home_score),
                'away_score': float(away_score),
                'time_remaining': time_remaining,
                'starttype_encoded': int(st_enc),
                'player_quality_home': pq_home,
                'player_quality_away': pq_away,
            })

    timeline = pd.DataFrame(timeline_rows).sort_values(['game_id', 'event_time']).reset_index(drop=True)
    if timeline.empty: raise ValueError('No PBP timeline rows were built')
    timeline['latent_time'] = timeline['event_time'] + pd.to_timedelta(LATENCY_BUFFER_MS, unit='ms')
    tipoff_lookup = timeline.groupby('game_id', as_index=False)['event_time'].min().rename(columns={'event_time': 'tipoff_time'})
    timeline = timeline.merge(tipoff_lookup, on='game_id', how='left')
    return timeline


def _safe_json(value: Any) -> Any:
    if isinstance(value, (list, dict)): return value
    if value is None or (isinstance(value, float) and math.isnan(value)): return None
    if isinstance(value, str):
        text = value.strip()
        if not text: return None
        try: return json.loads(text)
        except Exception:
            try: return ast.literal_eval(text)
            except Exception: return None
    return None


def attach_game_ids(frame: pd.DataFrame, mapping: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    direct_col = _candidate_column(list(out.columns), ['game_id'])
    if direct_col is not None:
        out['game_id'] = out[direct_col].astype(str)
        if 'yes_team_side' not in out.columns:
            out['yes_team_side'] = 'home'
        if 'resolved_market_id' not in out.columns:
            out['resolved_market_id'] = out['market_id'].astype(str)
        return out[out['game_id'].notna()].copy()

    if mapping is None or mapping.empty:
        raise ValueError('MARKET_LOOKUP is empty; unable to attach game IDs from PMXT data')

    lookup = mapping.drop_duplicates(subset=['lookup_key']).set_index('lookup_key')
    out['game_id'] = pd.NA
    out['yes_team_side'] = pd.NA
    out['resolved_market_id'] = pd.NA

    for key_col in ['market_id', 'event_slug', 'slug', 'market']:
        if key_col not in out.columns:
            continue
        key_values = out[key_col].astype(str)
        unresolved = out['game_id'].isna()
        out.loc[unresolved, 'game_id'] = key_values[unresolved].map(lookup['game_id'])
        out.loc[unresolved, 'yes_team_side'] = key_values[unresolved].map(lookup['yes_team_side'])
        out.loc[unresolved, 'resolved_market_id'] = key_values[unresolved].map(lookup['market_id'])

    out = out[out['game_id'].notna()].copy()
    out['game_id'] = out['game_id'].astype(str)
    out['yes_team_side'] = out['yes_team_side'].fillna('home').astype(str)
    out['resolved_market_id'] = out['resolved_market_id'].fillna(out['market_id']).astype(str)
    return out


def join_ticks_with_timeline(frame: pd.DataFrame, timeline: pd.DataFrame) -> pd.DataFrame:
    ticks = frame.copy()
    ticks['timestamp'] = pd.to_datetime(ticks['timestamp'], utc=True, errors='coerce')
    ticks['game_id'] = ticks['game_id'].astype(str)
    ticks = ticks[ticks['timestamp'].notna()].sort_values(['game_id', 'timestamp']).reset_index(drop=True)
    merged = pd.merge_asof(
        ticks,
        timeline.sort_values(['game_id', 'latent_time']),
        left_on='timestamp',
        right_on='latent_time',
        by='game_id',
        direction='backward',
        allow_exact_matches=True,
    )
    return merged[merged['timestamp'] >= merged['tipoff_time']].copy()

In [ ]:
def parse_levels(raw_levels: Any, descending: bool) -> list[tuple[float, float]]:
    payload = _safe_json(raw_levels)
    normalized = []
    for level in payload or []:
        price, size = None, None
        if isinstance(level, dict):
            price = level.get('price') or level.get('px') or level.get('p')
            size = level.get('size') or level.get('sz') or level.get('quantity') or level.get('q')
        elif isinstance(level, (list, tuple)) and len(level) >= 2:
            price, size = level[0], level[1]
        try:
            price, size = float(price), float(size)
        except (TypeError, ValueError):
            continue
        if price <= 0 or price >= 1 or size <= 0: continue
        normalized.append((float(price), float(size)))
    normalized.sort(key=lambda item: item[0], reverse=descending)
    return normalized

class OrderBookState:
    def __init__(self) -> None:
        self.bids: dict[float, float] = {}
        self.asks: dict[float, float] = {}

    def _apply_side(self, side: str, levels: list[tuple[float, float]], replace: bool) -> None:
        book = self.bids if side == 'bids' else self.asks
        if replace: book.clear()
        for price, size in levels:
            if size <= 0: book.pop(price, None)
            else: book[float(price)] = float(size)

    def update(self, record: dict[str, Any]) -> None:
        event_type = str(record.get('event_type') or record.get('type') or '').lower()
        bids = parse_levels(record.get('bids'), descending=True)
        asks = parse_levels(record.get('asks'), descending=False)
        is_snapshot = event_type == 'book_snapshot'
        
        if bids: self._apply_side('bids', bids, replace=is_snapshot)
        elif is_snapshot: self.bids.clear()
        if asks: self._apply_side('asks', asks, replace=is_snapshot)
        elif is_snapshot: self.asks.clear()
            
        for change in _safe_json(record.get('price_change')) or []:
            if isinstance(change, dict):
                side = str(change.get('side') or change.get('book') or '').lower()
                price = change.get('price') or change.get('px')
                size = change.get('size') or change.get('sz') or 0
            elif isinstance(change, (list, tuple)) and len(change) >= 3:
                side, price, size = change[0], change[1], change[2]
            else:
                continue
            try:
                price, size = float(price), float(size)
            except (TypeError, ValueError): continue
            
            target = self.bids if 'bid' in side else self.asks
            if size <= 0: target.pop(price, None)
            else: target[price] = size

    def best_bid(self) -> tuple[float | None, float]:
        if not self.bids: return None, 0.0
        price = max(self.bids)
        return price, float(self.bids[price])

    def best_ask(self) -> tuple[float | None, float]:
        if not self.asks: return None, 0.0
        price = min(self.asks)
        return price, float(self.asks[price])

    def vwap_buy(self, stake_usdc: float) -> tuple[float | None, float]:
        remaining, total_cost, total_shares = float(stake_usdc), 0.0, 0.0
        for price in sorted(self.asks):
            size = float(self.asks[price])
            if size <= 0: continue
            level_cost = float(price) * size
            if level_cost <= remaining:
                total_cost += level_cost
                total_shares += size
                remaining -= level_cost
            else:
                shares = remaining / float(price)
                total_cost += remaining
                total_shares += shares
                remaining = 0.0
                break
        if total_shares <= 0 or remaining > 1e-6: return None, total_shares
        return total_cost / total_shares, total_shares

@dataclass
class MarketPosition:
    market_id: str
    game_id: Any
    yes_shares: float = 0.0
    no_shares: float = 0.0
    yes_cost: float = 0.0
    no_cost: float = 0.0
    realized_pnl: float = 0.0
    cooldown_until: pd.Timestamp | None = None
    settled: bool = False
    settlement_value: float | None = None

@dataclass
class PortfolioState:
    initial_bankroll: float
    cash: float
    realized_pnl: float = 0.0
    positions: dict[str, MarketPosition] = field(default_factory=dict)

def total_bankroll(portfolio: PortfolioState) -> float:
    return float(portfolio.initial_bankroll + portfolio.realized_pnl)

def available_capital(portfolio: PortfolioState) -> float:
    return max(total_bankroll(portfolio) / PER_GAME_BANKROLL_DIVISOR, 0.0)

def fee_adjusted_kelly(probability: float, market_price: float, fee_rate: float = FEE_RATE, fraction: float = 0.25) -> float:
    if not (0 < market_price < 1) or not (0 < probability < 1): return 0.0
    b = (1.0 - market_price) / market_price
    q = 1.0 - probability
    full_kelly = ((probability * (1.0 - fee_rate) * b) - q) / b
    return round(max(full_kelly * fraction, 0.0), 6)

def ensure_position(portfolio: PortfolioState, market_id: str, game_id: Any) -> MarketPosition:
    if market_id not in portfolio.positions:
        portfolio.positions[market_id] = MarketPosition(market_id=market_id, game_id=game_id)
    return portfolio.positions[market_id]

def merge_positions(portfolio: PortfolioState, position: MarketPosition) -> None:
    paired = min(position.yes_shares, position.no_shares)
    if paired <= 0: return
    yes_release_cost = position.yes_cost * (paired / position.yes_shares) if position.yes_shares > 0 else 0.0
    no_release_cost = position.no_cost * (paired / position.no_shares) if position.no_shares > 0 else 0.0
    position.yes_shares -= paired
    position.no_shares -= paired
    position.yes_cost -= yes_release_cost
    position.no_cost -= no_release_cost
    realized = paired - yes_release_cost - no_release_cost
    portfolio.cash += paired
    portfolio.realized_pnl += realized
    position.realized_pnl += realized

def execute_buy(portfolio: PortfolioState, position: MarketPosition, book: OrderBookState, side: str, stake_usdc: float) -> dict[str, float] | None:
    if stake_usdc <= 0: return None
    spend = min(float(stake_usdc), float(portfolio.cash))
    if spend <= 0: return None
    raw_vwap, raw_shares = book.vwap_buy(spend)
    if raw_vwap is None or raw_shares <= 0: return None
    effective_price = min(max(float(raw_vwap) + ADVERSE_SELECTION_PENALTY, 0.001), 0.999)
    shares = spend / effective_price
    portfolio.cash -= spend
    if side == 'YES':
        position.yes_shares += shares
        position.yes_cost += spend
    else:
        position.no_shares += shares
        position.no_cost += spend
    merge_positions(portfolio, position)
    return {'spend': spend, 'price': effective_price, 'shares': shares}

def settle_position(portfolio: PortfolioState, position: MarketPosition, outcome_yes: bool) -> dict[str, float] | None:
    if position.settled: return None
    winning_shares = position.yes_shares if outcome_yes else position.no_shares
    total_cost = position.yes_cost + position.no_cost
    proceeds = float(winning_shares)
    gross_profit = proceeds - total_cost
    fee = max(gross_profit, 0.0) * FEE_RATE
    net_profit = gross_profit - fee
    portfolio.cash += proceeds - fee
    portfolio.realized_pnl += net_profit
    position.realized_pnl += net_profit
    position.yes_shares, position.no_shares, position.yes_cost, position.no_cost = 0.0, 0.0, 0.0, 0.0
    position.settled = True
    position.settlement_value = 1.0 if outcome_yes else 0.0
    return {'proceeds': proceeds, 'gross_profit': gross_profit, 'fee': fee, 'net_profit': net_profit}

In [ ]:
def normalize_pmxt_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [str(c) for c in out.columns]

    timestamp_col = _candidate_column(list(out.columns), ['timestamp', 'event_time', 'ts', 'created_at'])
    market_col = _candidate_column(list(out.columns), ['market_id', 'market', 'slug', 'event_slug'])
    event_col = _candidate_column(list(out.columns), ['event_type', 'type'])
    slug_col = _candidate_column(list(out.columns), ['slug'])
    event_slug_col = _candidate_column(list(out.columns), ['event_slug'])
    market_alias_col = _candidate_column(list(out.columns), ['market'])

    if timestamp_col is None or market_col is None:
        raise ValueError('PMXT chunk is missing a timestamp or market identifier column')

    out['timestamp'] = pd.to_datetime(out[timestamp_col], utc=True, errors='coerce')
    out['market_id'] = out[market_col].astype(str)
    out['event_type'] = out[event_col].astype(str) if event_col is not None else 'price_change'
    if slug_col is not None:
        out['slug'] = out[slug_col].astype(str)
    if event_slug_col is not None:
        out['event_slug'] = out[event_slug_col].astype(str)
    if market_alias_col is not None:
        out['market'] = out[market_alias_col].astype(str)

    for col in ['bids', 'asks', 'price_change', 'resolution']:
        if col not in out.columns: out[col] = None
    return out


def iter_pmxt_chunks(paths: list[Path], chunksize: int = CHUNK_SIZE):
    for path in paths:
        parquet_file = pq.ParquetFile(path)
        for batch in parquet_file.iter_batches(batch_size=chunksize):
            yield path, batch.to_pandas()


def _extract_resolution(record: dict[str, Any]) -> bool | None:
    event_type = str(record.get('event_type') or '').lower()
    if event_type not in {'condition_resolved', 'market_resolution'} and record.get('resolution') in (None, '', {}, []):
        return None
    resolution = record.get('resolution')
    payload = _safe_json(resolution) if not isinstance(resolution, dict) else resolution
    winner = str(payload.get('winner') or payload.get('outcome') or payload.get('result') or '').lower() if isinstance(payload, dict) else str(resolution).lower()
    if winner in {'yes', '1', 'true', 'home'}: return True
    if winner in {'no', '0', 'false', 'away'}: return False
    return None


def evaluate_signal(probability_yes: float, position: MarketPosition, portfolio: PortfolioState, book: OrderBookState) -> tuple[str, float] | None:
    best_ask, ask_size = book.best_ask()
    best_bid, bid_size = book.best_bid()
    if best_ask is None or best_bid is None: return None

    if float(best_ask) * float(ask_size) < MIN_RESTING_DEPTH_USDC or float(best_bid) * float(bid_size) < MIN_RESTING_DEPTH_USDC:
        return None

    yes_fraction = fee_adjusted_kelly(probability_yes, float(best_ask))
    no_fraction = fee_adjusted_kelly(1.0 - probability_yes, max(min(1.0 - float(best_bid), 0.999), 0.001))

    if yes_fraction <= 0 and no_fraction <= 0: return None

    if yes_fraction >= no_fraction:
        target = available_capital(portfolio) * yes_fraction
        delta = target - position.yes_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'YES', float(delta)
    else:
        target = available_capital(portfolio) * no_fraction
        delta = target - position.no_cost
        if delta > MIN_TARGET_DELTA_USDC and (delta / max(target, 1.0)) > MIN_DIVERGENCE_RATIO:
            return 'NO', float(delta)
    return None


def split_paths(paths: list[Path], holdout_pct: float = HOLDOUT_PCT) -> tuple[list[Path], list[Path]]:
    ordered = sorted(paths)
    if not ordered: return [], []
    cutoff = min(max(1, int(len(ordered) * (1.0 - holdout_pct))), len(ordered))
    return ordered[:cutoff], ordered[cutoff:]


def run_backtest(paths: list[Path], timeline: pd.DataFrame, market_lookup: pd.DataFrame, model: Any, feature_cols: list[str] | None, initial_bankroll: float = INITIAL_BANKROLL) -> dict[str, Any]:
    portfolio = PortfolioState(initial_bankroll=initial_bankroll, cash=initial_bankroll)
    books: dict[str, OrderBookState] = {}
    trades, equity = [], []

    for path, chunk in iter_pmxt_chunks(paths):
        joined = join_ticks_with_timeline(attach_game_ids(normalize_pmxt_frame(chunk), market_lookup), timeline)

        for record in joined.sort_values('timestamp').to_dict('records'):
            market_id = str(record.get('resolved_market_id') or record['market_id'])
            position = ensure_position(portfolio, market_id, record['game_id'])
            book = books.setdefault(market_id, OrderBookState())
            book.update(record)

            resolution = _extract_resolution(record)
            if resolution is not None:
                settlement = settle_position(portfolio, position, resolution)
                if settlement:
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': 'SETTLE_YES' if resolution else 'SETTLE_NO', **settlement})
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            if position.settled or (record.get('time_remaining') is not None and float(record['time_remaining']) <= 0):
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            if position.cooldown_until is not None and pd.Timestamp(record['timestamp']) < position.cooldown_until:
                equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})
                continue

            adv_ctx = None
            if USE_ADVANCED_FEATURES:
                adv_ctx = {'starttype_encoded': int(record.get('starttype_encoded', STARTTYPE_FALLBACK) or STARTTYPE_FALLBACK), 'player_quality_home': float(record.get('player_quality_home', 0.0) or 0.0), 'player_quality_away': float(record.get('player_quality_away', 0.0) or 0.0)}

            prob_home = predict_home_win_prob(model=model, feature_cols=feature_cols, home_score=int(record['home_score']), away_score=int(record['away_score']), period=int(record['period']), clock=str(record['clock']), advanced_ctx=adv_ctx)
            yes_team_side = str(record.get('yes_team_side') or 'home').lower()
            prob_yes = prob_home if yes_team_side == 'home' else 1.0 - prob_home

            signal = evaluate_signal(prob_yes, position, portfolio, book)
            if signal:
                side, delta = signal
                fill = execute_buy(portfolio, position, book, side, delta)
                if fill:
                    position.cooldown_until = pd.Timestamp(record['timestamp']) + pd.to_timedelta(COOLDOWN_MS, unit='ms')
                    trades.append({'timestamp': record['timestamp'], 'market_id': market_id, 'game_id': record['game_id'], 'action': f'BUY_{side}', 'model_prob_home': prob_home, 'model_prob_yes': prob_yes, 'yes_team_side': yes_team_side, 'cash_after': portfolio.cash, **fill})
            equity.append({'timestamp': record['timestamp'], 'equity': portfolio.cash, 'path': str(path)})

    BACKTEST_STATE_PATH.write_text(json.dumps({'initial_bankroll': portfolio.initial_bankroll, 'cash': portfolio.cash, 'realized_pnl': portfolio.realized_pnl, 'positions': {m: asdict(p) for m, p in portfolio.positions.items()}}, default=str, indent=2), encoding='utf-8')
    return {'portfolio': portfolio, 'trades': pd.DataFrame(trades), 'equity_curve': pd.DataFrame(equity), 'state_path': str(BACKTEST_STATE_PATH)}


def plot_equity_curve(result: dict[str, Any], label: str) -> None:
    equity = result['equity_curve'].copy()
    if equity.empty: return
    sampled = equity.iloc[::max(len(equity) // min(2000, len(equity)), 1)].copy()
    plt.figure(figsize=(12, 4))
    plt.plot(sampled['timestamp'], sampled['equity'])
    plt.title(f'Equity Curve — {label}')
    plt.xlabel('Timestamp')
    plt.ylabel('USDC')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
model, feature_cols = load_model(MODEL_PATH)
if model is None: raise FileNotFoundError(f'Model not found at {MODEL_PATH}')

df_nbastats = load_data_safe(NBA_SEASONS, 'nbastats')
df_pbpstats = load_data_safe(NBA_SEASONS, 'pbpstats') if USE_ADVANCED_FEATURES else None
player_ratings = load_player_ratings(USE_ADVANCED_FEATURES)
timeline = prepare_pbp_timeline(df_nbastats, df_pbpstats, player_ratings)
game_catalog = build_game_catalog(df_nbastats)

pmxt_paths = sorted(PMXT_ROOT.glob(PMXT_GLOB))
if not pmxt_paths: raise FileNotFoundError(f'No PMXT files found under {PMXT_ROOT} matching {PMXT_GLOB}')

pmxt_start_date, pmxt_end_date = infer_pmxt_date_range(pmxt_paths)
if pmxt_start_date is None or pmxt_end_date is None:
    dated_games = game_catalog['game_date'].dropna()
    if dated_games.empty:
        raise ValueError('Unable to infer a date range from PMXT files or game catalog')
    pmxt_start_date = dated_games.min().strftime('%Y-%m-%d')
    pmxt_end_date = dated_games.max().strftime('%Y-%m-%d')

print(f'PMXT date range: {pmxt_start_date} -> {pmxt_end_date}')

lower_game_date = pd.Timestamp(pmxt_start_date) - pd.Timedelta(days=MARKET_DATE_BUFFER_DAYS)
upper_game_date = pd.Timestamp(pmxt_end_date) + pd.Timedelta(days=MARKET_DATE_BUFFER_DAYS)
relevant_games = game_catalog[(game_catalog['game_date'].notna()) & (game_catalog['game_date'] >= lower_game_date) & (game_catalog['game_date'] <= upper_game_date)].copy()
if relevant_games.empty:
    raise ValueError('No games in the backtest catalog fall within the PMXT date range')

cache_key = build_lookup_cache_key(pmxt_start_date, pmxt_end_date, NBA_SEASONS, len(pmxt_paths))
cache_paths = lookup_cache_paths(cache_key)
print(f'Lookup cache key: {cache_key}')

cached_lookup = cached_matches = cached_diagnostics = None
if REUSE_SAVED_LOOKUP:
    cached_lookup, cached_matches, cached_diagnostics = load_lookup_cache(cache_key)

if cached_lookup is not None and cached_matches is not None and cached_diagnostics is not None:
    MARKET_LOOKUP = cached_lookup
    MARKET_MATCHES = cached_matches
    MATCH_DIAGNOSTICS = cached_diagnostics
    print(f'Loaded cached market lookup from: {cache_paths["lookup"]}')
else:
    gamma_markets_df = search_nba_markets(pmxt_start_date, pmxt_end_date, limit=100, closed=True, active=True, buffer_days=MARKET_DATE_BUFFER_DAYS)
    if gamma_markets_df.empty:
        raise ValueError(f'No Gamma NBA markets found between {pmxt_start_date} and {pmxt_end_date}')

    MARKET_MATCHES, MATCH_DIAGNOSTICS = match_gamma_markets_to_games(gamma_markets_df, relevant_games, buffer_days=MARKET_DATE_BUFFER_DAYS)
    if MARKET_MATCHES.empty:
        print('No strict market matches found. Showing diagnostics...')
        display(MATCH_DIAGNOSTICS.get('unmatched_markets', pd.DataFrame()).head(20))
        display(MATCH_DIAGNOSTICS.get('unmatched_games', pd.DataFrame()).head(20))
        display(MATCH_DIAGNOSTICS.get('ambiguous_candidates', pd.DataFrame()).head(20))
        raise ValueError('No Gamma markets could be matched to the backtest game catalog for the requested date range')

    MARKET_LOOKUP = build_market_lookup(MARKET_MATCHES)
    save_lookup_cache(cache_key, MARKET_LOOKUP, MARKET_MATCHES, MATCH_DIAGNOSTICS)
    print(f'Saved market lookup cache to: {cache_paths["lookup"]}')

MARKET_TO_GAME = dict(zip(MARKET_MATCHES['market_id'].astype(str), MARKET_MATCHES['game_id'].astype(str)))

print(f"Matched {MARKET_MATCHES['market_id'].nunique()} Gamma markets to {MARKET_MATCHES['game_id'].nunique()} backtest games")
display_cols = ['game_date', 'game_id', 'home_team', 'away_team', 'market_id', 'event_title', 'question', 'yes_team_side', 'score', 'score_gap', 'time_delta_hours']
display(MARKET_MATCHES[[col for col in display_cols if col in MARKET_MATCHES.columns]].head(20))

unmatched_games_df = MATCH_DIAGNOSTICS.get('unmatched_games', pd.DataFrame())
unmatched_markets_df = MATCH_DIAGNOSTICS.get('unmatched_markets', pd.DataFrame())
ambiguous_candidates_df = MATCH_DIAGNOSTICS.get('ambiguous_candidates', pd.DataFrame())

print(f'Unmatched backtest games: {len(unmatched_games_df)}')
if not unmatched_games_df.empty:
    display(unmatched_games_df.head(20))

print(f'Unmatched Gamma markets: {len(unmatched_markets_df)}')
if not unmatched_markets_df.empty:
    display(unmatched_markets_df[[col for col in ['event_start_ts', 'market_id', 'event_title', 'question', 'liquidity'] if col in unmatched_markets_df.columns]].head(20))

print(f'Ambiguous market candidates: {len(ambiguous_candidates_df)}')
if not ambiguous_candidates_df.empty:
    display(ambiguous_candidates_df[[col for col in ['market_id', 'event_title', 'question', 'game_id', 'home_team', 'away_team', 'score', 'time_delta_hours'] if col in ambiguous_candidates_df.columns]].head(20))

print(f"Beginning Backtest Iteration on {len(pmxt_paths)} files")
tuning_result = run_backtest(pmxt_paths, timeline, MARKET_LOOKUP, model, feature_cols, initial_bankroll=INITIAL_BANKROLL)

print(f"Ending Balance: {tuning_result['portfolio'].cash:.2f}")
plot_equity_curve(tuning_result, 'full')